# Introduction

The goal is to verify the alignment of the sample peaks in a batch and make sure batch aggregated peaks are correct.

**WARNING**: Currently works only for batches with one ionization mode. Making it work for multiple ionization modes would require tracking the correspondence of batch peaks to sample peak cluster numbers per ionization mode.

# Database Initialization

In [ ]:
from mascope_backend.db import init_db

await init_db()

# Data Collection

In [ ]:
import pandas as pd
import numpy as np
from mascope_signal.compute import align_peak_collection, AGGREGATION_WINDOW_FACTOR
from mascope_tools.alignment.calibration import Spectra
from mascope_backend.api.controllers.sample.batches.sample_batches_controller import (
    _collect_spectra_per_ionization_mode,
    get_sample_batch_peaks,
)

sample_batch_id = "EHcWyasdCruYN9JT"

# --- Collect, align, and cluster sample peaks --- #
# Collect sample peaks per ionization mode
sample_peaks, _ = await _collect_spectra_per_ionization_mode(sample_batch_id)
# Form Spectra objects (collections of CentroidedSpectrum with timestamps)
sample_peak_collections = {
    mode: Spectra(sp, timestamps=np.arange(len(sp)))
    for mode, sp in sample_peaks.items()
}
# Align sample peak collections
aligned_sample_peak_collections = {
    mode: align_peak_collection(spc)[0] for mode, spc in sample_peak_collections.items()
}
# Compute clusters of aligned peaks from which batch peaks are aggregated
aligned_peak_clusters = {
    mode: spc._cluster_and_map_peaks(AGGREGATION_WINDOW_FACTOR)
    for mode, spc in aligned_sample_peak_collections.items()
}
# Concatenate and sort aligned peaks from all ionization modes
aligned_peaks = (
    pd.concat(aligned_peak_clusters.values())
    .sort_values(by="mz")
    .reset_index(drop=True)
)

# --- Fetch batch peaks using the controller --- #
batch_peaks = (
    await get_sample_batch_peaks(
        sample_batch_id,
    )
)["data"]

# Visualization

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

current_idx = 0
vlm_min = batch_peaks["min_aligned_mz"]
vlm_max = batch_peaks["max_aligned_mz"]


def plot_peaks(idx):
    clear_output(wait=True)
    mz = batch_peaks["mzs"][idx]
    cluster_mz = aligned_peaks["mz"][aligned_peaks["cluster_id"] == idx]
    cluster_intensity = aligned_peaks["intensity"][aligned_peaks["cluster_id"] == idx]

    plt.figure(figsize=(10, 6))
    plt.axvline(x=mz, color="#882020", linestyle="-", linewidth=3, label="Batch Peak")
    plt.scatter(
        x=cluster_mz,
        y=cluster_intensity,
        s=100,
        alpha=0.5,
        color="green",
        label="Aligned Sample Peaks",
    )
    plt.title(
        f'Batch Peak {idx+1}/{len(batch_peaks["mzs"])}: {mz:.4f} m/z. Alignment range: {vlm_min:.2f}-{vlm_max:.2f} m/z'
    )
    plt.xlabel("m/z")
    plt.ylabel("Intensity")
    # plt.xlim(min(cluster_mz) - 0.00001, max(cluster_mz) + 0.00001)
    plt.legend()
    plt.show()
    display(widgets.HBox([prev_button, next_button]))


def on_next(b):
    global current_idx
    if current_idx < len(batch_peaks["mzs"]) - 1:
        current_idx += 1
    plot_peaks(current_idx)


def on_prev(b):
    global current_idx
    if current_idx > 0:
        current_idx -= 1
    plot_peaks(current_idx)


prev_button = widgets.Button(description="Previous")
next_button = widgets.Button(description="Next")
prev_button.on_click(on_prev)
next_button.on_click(on_next)

plot_peaks(current_idx)

# Algorithm Details

## Sum Spectrum Computation and Clustering
The `compute_sum_spectrum` method in `Spectra` aggregates total peaks from a collection of centroided spectra. This process involves clustering peaks across all spectra and aggregating their properties:

- Flatten all peaks from each centroided spectrum keeping track of the spectrum index.
- Sort by m/z
- Cluster by sliding a window across the sorted peaks. For each unassigned peak:
    - Compute its peak width (FWHM) as peak_width = m/z / resolution.
    - Define a clustering window: window = peak_width * window_factor
    - All consecutive peaks within this window (with m/z less than start_mz + window) are assigned to the same cluster.
    - Each cluster receives a unique integer cluster_id.
- For each cluster aggregate:
    - Total intensity: Sum of intensities of all peaks in the cluster.
    - Weighted average m/z: weighted_avg_mz = sum(mz_i * intensity_i) / sum(intensity_i)
    - Weighted average resolution: weighted_avg_res = sum(resolution_i * intensity_i) / sum(intensity_i)
    - Total signal-to-noise ratio: total_snr = sum(S) / sqrt(sum((S/R)^2)), where S = intensity, R = signal-to-noise for each peak.
    - Peak IDs: list of all peak IDs in the cluster.
- Average (optional, by dividing by number of spectra)
- The aggregated cluster properties are used to construct a new `CentroidedSpectrum` object, representing the sum (or average) spectrum.